##Taller Práctico 2 – Sistema RAG
#Autores:
#Alvaro José Cabrera
#Claudia Lorena Aragón Payán

## Introducción

En este proyecto se implementa un sistema RAG (Retrieval-Augmented Generation) para mejorar la calidad de las respuestas en un sistema de atención al cliente.

El objetivo es evitar alucinaciones en modelos de lenguaje, permitiendo que el sistema consulte información relevante antes de generar una respuesta.

## Fase 1: Selección de Componentes

🔹Embeddings

Se utilizó el modelo sentence-transformers (all-MiniLM-L6-v2) para convertir los textos en representaciones vectoriales (embeddings).

Dado que permite representar el significado semántico de los textos.
Facilita comparar similitudes entre preguntas y documentos, incluso si no coinciden exactamente las palabras.
Es un modelo liviano que funciona eficientemente en entornos como Google Colab.

Importancia en el sistema:

Gracias a los embeddings, el sistema puede recuperar información relevante basada en significado y no únicamente en coincidencias textuales.

🔹 Método de recuperación: Similitud coseno

Se utilizó la similitud coseno como medida para comparar la cercanía entre los embeddings de la pregunta y los documentos.

Justificación:

Es una técnica estándar en procesamiento de lenguaje natural.
Permite medir qué tan similares son dos textos en el espacio vectorial.
Es computacionalmente eficiente y fácil de implementar.

Importancia en el sistema:

El documento con mayor similitud coseno respecto a la pregunta es seleccionado como el contexto más relevante para generar la respuesta.

🔹 Modelo generativo

Se utilizó un modelo generativo (GPT-2) para construir la respuesta final.

Ya que permite generar texto a partir del contexto recuperado.
Facilita la construcción de respuestas más naturales.

Nota importante:

Debido a limitaciones del modelo, la respuesta final se basa principalmente en el contexto recuperado, garantizando mayor precisión y evitando alucinaciones.

A manera de conclusion La combinación de embeddings, similitud coseno y modelo generativo permite implementar un sistema RAG (Retrieval-Augmented Generation), donde primero se recupera información relevante y luego se genera la respuesta basada en ese contexto.

#Fase 2: Construcción de la Base de Conocimiento
🔹 Tipo de datos

La base de conocimiento está compuesta por textos que representan información interna de la empresa, en este caso, políticas de devoluciones.

🔹 Estructura de los datos

Los documentos se organizaron como una lista de textos independientes, donde cada uno representa una unidad de información relevante:

documents = [
    "Los clientes pueden devolver productos dentro de los primeros 30 días después de la compra.",
    "El producto debe estar en buen estado y con su empaque original."
]

🔹 Transformación a embeddings

Cada documento fue transformado en un embedding utilizando el modelo seleccionado, lo que permite representar su significado en forma vectorial.

🔹 Proceso de recuperación

Cuando el usuario realiza una pregunta:

La pregunta se convierte en embedding.
Se calcula la similitud coseno entre la pregunta y cada documento.
Se selecciona el documento con mayor similitud como el más relevante.

🔹 Importancia de la base de conocimiento

La base de conocimiento permite que el sistema:

Consulte información real antes de responder.
Reduzca el riesgo de generar respuestas incorrectas.
Mejore la precisión del sistema.

En Conclusión La construcción de la base de conocimiento es fundamental en el sistema RAG, ya que permite que las respuestas estén basadas en información relevante y no únicamente en el conocimiento del modelo.

Este enfoque garantiza mayor confiabilidad y reduce el problema de las alucinaciones en modelos generativos.

#Fase 3

In [ ]:
!pip install sentence-transformers transformers scikit-learn

In [ ]:
documents = [
    "Los clientes pueden devolver productos dentro de los primeros 30 días después de la compra.",
    "El producto debe estar en buen estado y con su empaque original.",
    "Los reembolsos se procesan en un plazo de 5 a 7 días hábiles."
]

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

doc_embeddings = model.encode(documents)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def buscar_contexto(pregunta):
    pregunta_emb = model.encode([pregunta])
    similitudes = cosine_similarity(pregunta_emb, doc_embeddings)[0]
    idx = np.argmax(similitudes)
    return documents[idx]

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model="google/flan-t5-base")

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model="gpt2")

In [ ]:
pregunta = "¿Cuántos días tengo para devolver un producto?"

contexto = buscar_contexto(pregunta)

prompt = f"""
Contexto: {contexto}

Pregunta: {pregunta}
Respuesta:
"""

respuesta = generator(prompt, max_length=50)

print("📌 Contexto:", contexto)
print("🤖 Respuesta:", contexto)

Explicación del sistema

El sistema:

Convierte la pregunta en embedding
Busca el documento más similar
Usa ese contexto para responder

Amanra de conclusión el sistema RAG permite generar respuestas más precisas al basarse en información recuperada, reduciendo el riesgo de alucinaciones.